# Segunda Parte: Torneo Algorítmico Avanzado y Modelos No Paramétricos

## Introducción a la Fase 2

Tras establecer un Baseline sólido mediante un modelo de **Regresión Lineal Múltiple (MLR)** en la Fase 1 —donde mitigamos de manera exitosa la multicolinealidad estructural extrema a través de la Ingeniería de Características (creación de la variable `ritmo_promedio_30k`)—, el objetivo de esta segunda fase metodológica es llevar a cabo un **Torneo Algorítmico**. 

En esta etapa, dejaremos atrás las asunciones paramétricas estrictas (como la linealidad global) para implementar y evaluar una batería de algoritmos de alta complejidad y naturaleza no paramétrica. El reto principal es superar las métricas de nuestro Baseline actual: un **MAE de 4.69 minutos**, un **RMSE de 7.85 minutos** y un **R² de 96.9%**. 

Para garantizar la integridad del experimento y la reproducibilidad académica, todo el modelado, particionamiento de datos, validación cruzada y sintonización de hiperparámetros (*Grid Search*) se ejecutará bajo el riguroso framework orientado a objetos de la librería `mlr` en R.

---

## Paso 1: Configuración del Entorno y Entrenamiento del Árbol de Regresión (CART)

Los árboles de regresión son modelos no paramétricos que particionan recursivamente el espacio de características hiperdimensional en subregiones más simples (**nodos**). A diferencia de la Regresión Lineal Múltiple de nuestra Fase 1, que asume una relación paramétrica lineal global, el árbol predice la variable objetivo estimando la media de las observaciones que caen en cada nodo terminal (**hoja**).

En el ecosistema `mlr`, utilizaremos el algoritmo `rpart` debido a su robustez empírica para manejar tareas de regresión continua y su capacidad inherente para capturar interacciones no lineales sin requerir transformaciones previas exhaustivas.

In [2]:
# =====================================================================
# FASE 2: TORNEO ALGORÍTMICO AVANZADO - PREDICCIÓN DE TIEMPOS DE MARATÓN
# Contendiente 1: Árbol de Regresión (CART) Optimizado
# =====================================================================

# Cargar las librerías necesarias
library(mlr)
library(tidyverse)

# ---------------------------------------------------------------------
# 1. Ingesta de Datos y Particionamiento Estratégico (Hold-out 70/30)
# ---------------------------------------------------------------------
# Cargamos el dataset limpio exportado en la Fase 1
marathon_data <- readRDS("marathon_2015_clean.rds")

# Fijamos la semilla para garantizar la reproducibilidad y la justa comparativa
set.seed(2015)

# Particionamiento Hold-out: 70% Entrenamiento, 30% Prueba
train_indices <- sample(seq_len(nrow(marathon_data)), size = 0.7 * nrow(marathon_data))
train_data <- marathon_data[train_indices, ]
test_data <- marathon_data[-train_indices, ]

# ---------------------------------------------------------------------
# 2. Arquitectura mlr: Definición de la Tarea y el Aprendiz
# ---------------------------------------------------------------------
# mlr requiere que encapsulemos nuestros datos y la variable objetivo en un objeto 'Task'
train_task <- makeRegrTask(data = train_data, target = "official_time_min")
test_task <- makeRegrTask(data = test_data, target = "official_time_min")

# Instanciamos nuestro primer aprendiz (Learner) no paramétrico
learner_rpart <- makeLearner("regr.rpart")

# ---------------------------------------------------------------------
# 3. Protocolo de Evaluación y Espacio de Hiperparámetros
# ---------------------------------------------------------------------
# Definimos nuestra estrategia de Validación Cruzada k-fold (k=5)
# Esto particiona internamente el train_data para la sintonización
cv_strategy <- makeResampleDesc("CV", iters = 5)

# Definimos el espacio de búsqueda de hiperparámetros con makeParamSet() [cite: 121]
# - cp (Complexity Parameter): Penaliza el tamaño del árbol (evita sobreajuste)
# - maxdepth: Profundidad máxima permitida del árbol
# - minsplit: Observaciones mínimas requeridas en un nodo para intentar dividirlo
tree_param_space <- makeParamSet(
  makeNumericParam("cp", lower = 0.001, upper = 0.05),
  makeIntegerParam("maxdepth", lower = 3, upper = 10),
  makeIntegerParam("minsplit", lower = 10, upper = 50)
)

# Definimos nuestro viejo amigo, la búsqueda en rejilla (Grid Search), utilizando makeTuneControlGrid() [cite: 122]
# Explorará combinaciones paramétricas para encontrar el óptimo global dentro del espacio definido
tune_control <- makeTuneControlGrid(resolution = 5)

# ---------------------------------------------------------------------
# 4. Sintonización (Tuning) y Entrenamiento del Modelo Final
# ---------------------------------------------------------------------
cat("Iniciando la sintonización de hiperparámetros vía Grid Search...\n")

# Para realizar el ajuste usamos tuneParams(). El primer argumento es el aprendizaje, y luego proporcionamos nuestra tarea, el método de validación cruzada, el espacio de hiperparámetros y el procedimiento de búsqueda[cite: 123, 124].
# Usaremos el Mean Absolute Error (MAE) como métrica principal para superar el Baseline (4.69 min),
# junto con el RMSE, ya que este tiene la ventaja de estar en la misma escala que nuestra variable de resultado y es muy interpretable[cite: 129].
tuned_tree <- tuneParams(
  learner = learner_rpart,
  task = train_task,
  resampling = cv_strategy,
  par.set = tree_param_space,
  control = tune_control,
  measures = list(mae, rmse, rsq)
)

cat("=== Hiperparámetros Óptimos Encontrados ===\n")
print(tuned_tree$x)

# Insertamos los hiperparámetros óptimos en un nuevo aprendiz definitivo
final_learner_rpart <- setHyperPars(learner_rpart, par.vals = tuned_tree$x)

# Entrenamos el modelo final con todos los datos de entrenamiento (Train Data)
final_tree_model <- train(final_learner_rpart, train_task)

cat("\nModelo de Árbol de Regresión entrenado exitosamente.\n")

Warning message in makeTask(type = type, data = data, weights = weights, blocking = blocking, :
"Provided data is not a pure data.frame but from class tbl_df, hence it will be converted."
Warning message in makeTask(type = type, data = data, weights = weights, blocking = blocking, :
"Provided data is not a pure data.frame but from class tbl_df, hence it will be converted."


Iniciando la sintonización de hiperparámetros vía Grid Search...


[Tune] Started tuning learner regr.rpart for parameter set:

            Type len Def        Constr Req Tunable Trafo
cp       numeric   -   - 0.001 to 0.05   -    TRUE     -
maxdepth integer   -   -       3 to 10   -    TRUE     -
minsplit integer   -   -      10 to 50   -    TRUE     -

With control class: TuneControlGrid

Imputation value: InfImputation value: InfImputation value: Inf

[Tune-x] 1: cp=0.001; maxdepth=3; minsplit=10

[Tune-y] 1: mae.test.mean=7.1887600,rmse.test.rmse=10.0165279,rsq.test.mean=0.9386778; time: 0.0 min

[Tune-x] 2: cp=0.0133; maxdepth=3; minsplit=10

[Tune-y] 2: mae.test.mean=7.1887600,rmse.test.rmse=10.0165279,rsq.test.mean=0.9386778; time: 0.0 min

[Tune-x] 3: cp=0.0255; maxdepth=3; minsplit=10

[Tune-y] 3: mae.test.mean=11.6413148,rmse.test.rmse=14.8039359,rsq.test.mean=0.8660488; time: 0.0 min

[Tune-x] 4: cp=0.0378; maxdepth=3; minsplit=10

[Tune-y] 4: mae.test.mean=11.6413148,rmse.test.rmse=14.8039359,rsq.test.mean=0.8660488; time: 0.0 min

[Tune-x

=== Hiperparámetros Óptimos Encontrados ===
$cp
[1] 0.001

$maxdepth
[1] 6

$minsplit
[1] 20


Modelo de Árbol de Regresión entrenado exitosamente.


### Sintonización de Hiperparámetros y Entrenamiento (Resultados)

El proceso de optimización topológica del árbol, ejecutado a través de una búsqueda exhaustiva en rejilla (*Grid Search*) validada de forma cruzada (5-folds), convergió en la siguiente configuración óptima de hiperparámetros:

* **Parámetro de Complejidad (`cp`) = 0.001**: Un valor cercano a cero que permitió ramificaciones sustanciales, reduciendo el sesgo estadístico pero manteniendo la regularización suficiente para podar divisiones estadísticamente irrelevantes.
* **Profundidad Máxima (`maxdepth`) = 6**: El algoritmo determinó que 6 niveles de profundidad bastaban para capturar las interacciones multivariables antes de incurrir en ruido (*overfitting*).
* **Mínimo de División (`minsplit`) = 20**: Se requirió una masa crítica mínima de 20 observaciones por nodo matriz para justificar una partición binaria.

A pesar de esta rigurosa calibración, la naturaleza predictiva del algoritmo CART —que genera predicciones de regresión en forma de función escalonada (promedios locales en las hojas terminales)— supone una desventaja teórica frente a algoritmos capaces de mapear superficies continuas cuando la variable objetivo presenta una alta correlación lineal latente, como la destilada en la Fase 1.

In [3]:
# ---------------------------------------------------------------------
# 5. Evaluación del Modelo de Árbol de Regresión en Test Data
# ---------------------------------------------------------------------
cat("\n=== EVALUACIÓN EN CONJUNTO DE PRUEBA (TEST DATA) ===\n")

# Usamos la función predict() nativa de mlr, pasando nuestro modelo final y el Test Task
tree_predictions <- predict(final_tree_model, test_task)

# Extraemos los valores reales (truth) y los predichos (response)
reales <- tree_predictions$data$truth
predichos <- tree_predictions$data$response

# Calculamos las métricas de rendimiento manualmente para mayor control y rigor
mae_tree <- mean(abs(reales - predichos))
rmse_tree <- sqrt(mean((reales - predichos)^2))

# Cálculo formal del R-cuadrado (R²)
ss_total <- sum((reales - mean(reales))^2)
ss_residual <- sum((reales - predichos)^2)
rsq_tree <- 1 - (ss_residual / ss_total)

# Imprimimos la comparativa final del Torneo
cat(sprintf("-> MAE del Árbol (CART)  : %.4f minutos\n", mae_tree))
cat(sprintf("-> RMSE del Árbol (CART) : %.4f minutos\n", rmse_tree))
cat(sprintf("-> R² del Árbol (CART)   : %.4f %%\n", rsq_tree * 100))
cat("\n--- RECORDATORIO DEL BASELINE (FASE 1) ---\n")
cat("-> MAE Baseline (MLR)    : 4.6900 minutos\n")
cat("-> RMSE Baseline (MLR)   : 7.8500 minutos\n")
cat("-> R² Baseline (MLR)     : 96.9000 %\n")


=== EVALUACIÓN EN CONJUNTO DE PRUEBA (TEST DATA) ===
-> MAE del Árbol (CART)  : 5.5703 minutos
-> RMSE del Árbol (CART) : 8.6955 minutos
-> R² del Árbol (CART)   : 95.3468 %

--- RECORDATORIO DEL BASELINE (FASE 1) ---
-> MAE Baseline (MLR)    : 4.6900 minutos
-> RMSE Baseline (MLR)   : 7.8500 minutos
-> R² Baseline (MLR)     : 96.9000 %


### Discusión de Resultados: Árbol de Regresión vs. Baseline

La evaluación del modelo CART optimizado sobre el conjunto de prueba aislado (Test Data) arrojó un Error Absoluto Medio (MAE) de **5.5703 minutos** y un R² de **95.34%**. Al comparar estas métricas contra nuestro Baseline de Regresión Lineal Múltiple (MAE de **4.6900 minutos**; R² de **96.90%**), se concluye empíricamente que el modelo no paramétrico basado en árboles no logró superar la aproximación paramétrica.

**Justificación Matemática:**
Este fenómeno obedece a la naturaleza topológica de los algoritmos CART. Mientras que la Regresión Lineal ajusta un hiperplano continuo que se alinea perfectamente con la alta colinealidad de nuestra variable sintética (`ritmo_promedio_30k`), el árbol de regresión aproxima la superficie de respuesta mediante funciones escalonadas constantes a trozos (*piecewise constant functions*). Esta discretización del espacio introduce un sesgo ineludible al intentar interpolar una variable objetivo estrictamente continua y fuertemente correlacionada, justificando la degradación del rendimiento de predicción.

---

## Paso 2: Implementación de k-Nearest Neighbors (k-NN) para Regresión

El segundo algoritmo a competir en nuestro torneo es **k-Nearest Neighbors (k-NN)**. Este es un algoritmo de aprendizaje perezoso (*lazy learning*) fundamentado en la premisa de que observaciones con características similares tenderán a poseer valores de respuesta similares. A diferencia del enfoque paramétrico, k-NN estima el tiempo de llegada de un corredor promediando los tiempos de los "k" corredores más cercanos a él en el espacio hiperdimensional de características. 

**Consideración Crítica: Estandarización de Distancias**
Dado que k-NN utiliza métricas de distancia (típicamente Distancia Euclidiana) para encontrar vecinos, es matemáticamente hiper-sensible a la magnitud de las variables. En nuestro conjunto de datos, una variable como `age` (que varía entre 18 y 80) dominaría geométricamente a la variable `ritmo_promedio_30k` (cuya varianza ocurre en decimales minúsculos), anulando su poder predictivo. Por mandato, debemos aplicar una normalización Z-score (centrado y escalado) antes de computar las distancias.

In [5]:
# =====================================================================
# Contendiente 2: k-Nearest Neighbors (k-NN) Adaptado para Regresión
# =====================================================================

# Asegúrate de tener instalado el paquete kknn
install.packages("kknn")
library(kknn)

cat("\n--- 1. Preprocesamiento: Estandarización de Variables ---\n")
# mlr permite normalizar todo el Task de manera segura. 
# Z-score normalization: restamos la media y dividimos por la desviación estándar.
train_task_norm <- normalizeFeatures(train_task, method = "standardize")
test_task_norm  <- normalizeFeatures(test_task, method = "standardize")

# --- 2. Definición del Aprendiz k-NN ---
learner_knn <- makeLearner("regr.kknn")

# --- 3. Espacio de Búsqueda de Hiperparámetros ---
# Exploraremos distintos valores de 'k' y dos tipos de cálculo de distancia:
# 'rectangular' = promedio simple (peso igual para todos los 'k' vecinos)
# 'optimal' = ponderación por distancia (vecinos más cerca pesan más en la predicción)
knn_param_space <- makeParamSet(
  makeIntegerParam("k", lower = 3, upper = 35),
  makeDiscreteParam("kernel", values = c("rectangular", "optimal"))
)

# Control de la búsqueda (Grid Search)
tune_control_knn <- makeTuneControlGrid()

cat("\n--- 4. Sintonización de Hiperparámetros (Grid Search k-NN) ---\n")
# OJO: Dependiendo del procesador, k-NN toma un poco más de tiempo porque 
# computa las distancias de todos contra todos en cada fold.
tuned_knn <- tuneParams(
  learner = learner_knn,
  task = train_task_norm,
  resampling = cv_strategy, # Mantenemos el mismo 5-fold CV de la fase anterior
  par.set = knn_param_space,
  control = tune_control_knn,
  measures = list(mae, rmse, rsq)
)

cat("\n=== Hiperparámetros Óptimos Encontrados para k-NN ===\n")
print(tuned_knn$x)

# --- 5. Entrenamiento del Modelo Definitivo ---
final_learner_knn <- setHyperPars(learner_knn, par.vals = tuned_knn$x)
final_knn_model <- train(final_learner_knn, train_task_norm)

cat("\nModelo k-NN entrenado y listo para evaluación.\n")

also installing the dependency 'igraph'




package 'igraph' successfully unpacked and MD5 sums checked
package 'kknn' successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\axurm\AppData\Local\Temp\Rtmps9LO7u\downloaded_packages


Warning message:
"package 'kknn' was built under R version 4.5.3"



--- 1. Preprocesamiento: Estandarización de Variables ---

--- 4. Sintonización de Hiperparámetros (Grid Search k-NN) ---


[Tune] Started tuning learner regr.kknn for parameter set:

           Type len Def              Constr Req Tunable Trafo
k       integer   -   -             3 to 35   -    TRUE     -
kernel discrete   -   - rectangular,optimal   -    TRUE     -

With control class: TuneControlGrid

Imputation value: InfImputation value: InfImputation value: Inf

[Tune-x] 1: k=3; kernel=rectangular

[Tune-y] 1: mae.test.mean=5.3150055,rmse.test.rmse=8.2933565,rsq.test.mean=0.9579338; time: 0.1 min

[Tune-x] 2: k=7; kernel=rectangular

[Tune-y] 2: mae.test.mean=4.9395463,rmse.test.rmse=7.7378238,rsq.test.mean=0.9634008; time: 0.1 min

[Tune-x] 3: k=10; kernel=rectangular

[Tune-y] 3: mae.test.mean=4.8370684,rmse.test.rmse=7.5933707,rsq.test.mean=0.9647614; time: 0.1 min

[Tune-x] 4: k=14; kernel=rectangular

[Tune-y] 4: mae.test.mean=4.7841199,rmse.test.rmse=7.5303647,rsq.test.mean=0.9653459; time: 0.1 min

[Tune-x] 5: k=17; kernel=rectangular

[Tune-y] 5: mae.test.mean=4.7555384,rmse.test.rmse=7.501359


=== Hiperparámetros Óptimos Encontrados para k-NN ===
$k
[1] 35

$kernel
[1] "optimal"


Modelo k-NN entrenado y listo para evaluación.


### Sintonización de Hiperparámetros y Entrenamiento (k-NN)

Dado que k-NN es un algoritmo de aprendizaje basado en instancias y dependiente de la escala (*scale-sensitive*), el primer paso consistió en aplicar una transformación de Estandarización Z-score (centrado en media 0 y varianza 1) a todo el espacio de características. Esto garantizó que variables con magnitudes dispares, como la Edad (`age`) y el Ritmo Promedio (`ritmo_promedio_30k`), contribuyeran equitativamente al cálculo de la Distancia Euclidiana.

Posteriormente, se ejecutó una Búsqueda en Rejilla (*Grid Search*) validada de forma cruzada (5-folds) para optimizar la topología de la vecindad. El algoritmo convergió en los siguientes hiperparámetros óptimos:
* **`k` (Número de vecinos) = 35**: Una vecindad amplia que actúa como un regularizador natural, reduciendo la varianza del modelo (suavizando la frontera de decisión) al promediar el ruido inherente a los tiempos individuales de los corredores.
* **`kernel` = optimal**: Una función de decaimiento por distancia (*distance weighting*) que otorga mayor peso ponderado a los vecinos más cercanos espacialmente dentro de los 35 seleccionados. Esta combinación permite aprovechar la estabilidad estadística de un vecindario grande sin sacrificar la precisión predictiva de las instancias más similares.

In [6]:
# ---------------------------------------------------------------------
# 6. Evaluación del Modelo k-NN en Test Data (Normalizado)
# ---------------------------------------------------------------------
cat("\n=== EVALUACIÓN EN CONJUNTO DE PRUEBA (TEST DATA) ===\n")

# Usamos predict() con el modelo k-NN y el Task de prueba NORMALIZADO
knn_predictions <- predict(final_knn_model, test_task_norm)

# Extraemos los valores reales y predichos
reales_knn <- knn_predictions$data$truth
predichos_knn <- knn_predictions$data$response

# Calculamos las métricas de rendimiento
mae_knn <- mean(abs(reales_knn - predichos_knn))
rmse_knn <- sqrt(mean((reales_knn - predichos_knn)^2))

# Cálculo del R-cuadrado (R²)
ss_total_knn <- sum((reales_knn - mean(reales_knn))^2)
ss_residual_knn <- sum((reales_knn - predichos_knn)^2)
rsq_knn <- 1 - (ss_residual_knn / ss_total_knn)

# Imprimimos la comparativa actualizada del Torneo
cat("--- TABLA DE POSICIONES DEL TORNEO ---\n")
cat("1. Baseline (MLR)       -> MAE: 4.6900 min | RMSE: 7.8500 min | R²: 96.90%\n")
cat(sprintf("2. k-NN (k=35, optimal) -> MAE: %.4f min | RMSE: %.4f min | R²: %.2f%%\n", mae_knn, rmse_knn, rsq_knn * 100))
cat("3. CART (Árbol)         -> MAE: 5.5703 min | RMSE: 8.6955 min | R²: 95.34%\n")


=== EVALUACIÓN EN CONJUNTO DE PRUEBA (TEST DATA) ===
--- TABLA DE POSICIONES DEL TORNEO ---
1. Baseline (MLR)       -> MAE: 4.6900 min | RMSE: 7.8500 min | R²: 96.90%
2. k-NN (k=35, optimal) -> MAE: 5.0035 min | RMSE: 8.1025 min | R²: 95.96%
3. CART (Árbol)         -> MAE: 5.5703 min | RMSE: 8.6955 min | R²: 95.34%


### Discusión de Resultados: k-NN vs. Baseline

La evaluación en el conjunto de prueba (Test Data) demostró que el modelo k-NN optimizado alcanzó un Error Absoluto Medio (MAE) de **5.0035 minutos** y un R² del **95.96%**. Si bien este enfoque geométrico superó significativamente al modelo topológico de árboles (CART), aún se queda rezagado frente al Baseline paramétrico (MLR), cuyo MAE se sostiene en **4.6900 minutos**.

**Justificación Matemática:**
El algoritmo k-NN, incluso con ponderación óptima de distancias y datos rigurosamente estandarizados, aproxima la superficie de respuesta mediante un suavizado local. Sin embargo, la Ingeniería de Características implementada en la Fase 1 reveló una estructura de datos latente gobernada por una correlación lineal extrema. Para mapear con precisión este tipo de distribuciones, un hiperplano continuo global (como el de la regresión lineal) resulta teóricamente más eficiente que el cálculo de distancias euclidianas locales, explicando así la supremacía estadística que el modelo paramétrico aún mantiene.

---

## Paso 3: Implementación de Máquinas de Soporte Vectorial (SVM)

Como tercer contendiente, introducimos las Máquinas de Soporte Vectorial adaptadas para regresión (Support Vector Regression - SVR). A diferencia de la regresión tradicional que penaliza cualquier error, SVR busca un hiperplano que contenga la mayor cantidad de observaciones dentro de un margen de tolerancia predefinido (tubo $\epsilon$). 

El mayor poder de las SVM radica en el **"Truco del Kernel"** (*Kernel Trick*). Si la relación entre los tiempos intermedios y el tiempo final de los maratonistas presenta curvaturas hiperdimensionales, un Kernel Radial (RBF) mapeará implícitamente los datos a un espacio de mayor dimensionalidad donde estas relaciones no lineales se vuelvan linealmente separables. Al igual que k-NN, este modelo requiere el uso imperativo del espacio de características normalizado (Z-score) para que los vectores de soporte no se vean sesgados por la magnitud de las variables.

In [7]:
# =====================================================================
# Contendiente 3: Máquinas de Soporte Vectorial (SVM / SVR)
# =====================================================================

# Asegúrate de instalar e1071 si no lo tienes
# install.packages("e1071")
library(e1071)

cat("\n--- 1. Definición del Aprendiz SVM ---\n")
# Usaremos regr.svm. mlr lo conecta automáticamente con Support Vector Regression
learner_svm <- makeLearner("regr.svm")

cat("\n--- 2. Espacio de Búsqueda de Hiperparámetros ---\n")
# Optimizaremos tres dimensiones críticas:
# 1. kernel: 'linear' (básicamente una regresión lineal con margen) vs 'radial' (no linealidad compleja)
# 2. cost (C): Regularización. Penalización por puntos fuera del margen de tolerancia.
# 3. gamma: Define la influencia geométrica de un solo ejemplo de entrenamiento en el kernel radial.
svm_param_space <- makeParamSet(
  makeDiscreteParam("kernel", values = c("linear", "radial")),
  makeNumericParam("cost", lower = 0.1, upper = 10),
  makeNumericParam("gamma", lower = 0.01, upper = 1)
)

# Utilizaremos Random Search con 10 iteraciones para salvaguardar tu memoria RAM y tiempo de procesamiento
tune_control_svm <- makeTuneControlRandom(maxit = 10)

cat("\n--- 3. Sintonización de Hiperparámetros (Random Search SVM) ---\n")
cat("Nota: SVM es intensivo computacionalmente. Esto puede tomar varios minutos...\n")

# Reutilizamos el train_task_norm (datos estandarizados) porque SVM es hiper-sensible a la escala
tuned_svm <- tuneParams(
  learner = learner_svm,
  task = train_task_norm,
  resampling = cv_strategy, 
  par.set = svm_param_space,
  control = tune_control_svm,
  measures = list(mae, rmse, rsq)
)

cat("\n=== Hiperparámetros Óptimos Encontrados para SVM ===\n")
print(tuned_svm$x)

# --- 4. Entrenamiento del Modelo Definitivo ---
final_learner_svm <- setHyperPars(learner_svm, par.vals = tuned_svm$x)
final_svm_model <- train(final_learner_svm, train_task_norm)

cat("\nModelo SVM entrenado exitosamente y listo para el ruedo.\n")


Adjuntando el paquete: 'e1071'


The following object is masked from 'package:ggplot2':

    element


The following object is masked from 'package:mlr':

    impute





--- 1. Definición del Aprendiz SVM ---

--- 2. Espacio de Búsqueda de Hiperparámetros ---

--- 3. Sintonización de Hiperparámetros (Random Search SVM) ---
Nota: SVM es intensivo computacionalmente. Esto puede tomar varios minutos...


[Tune] Started tuning learner regr.svm for parameter set:

           Type len Def        Constr Req Tunable Trafo
kernel discrete   -   - linear,radial   -    TRUE     -
cost    numeric   -   -     0.1 to 10   -    TRUE     -
gamma   numeric   -   -     0.01 to 1   -    TRUE     -

With control class: TuneControlRandom

Imputation value: InfImputation value: InfImputation value: Inf

[Tune-x] 1: kernel=linear; cost=9.51; gamma=0.95

[Tune-y] 1: mae.test.mean=4.4689420,rmse.test.rmse=7.1401587,rsq.test.mean=0.9689001; time: 4.9 min

[Tune-x] 2: kernel=radial; cost=5.55; gamma=0.435

[Tune-y] 2: mae.test.mean=4.4633784,rmse.test.rmse=7.1874478,rsq.test.mean=0.9684785; time: 3.1 min

[Tune-x] 3: kernel=linear; cost=5.62; gamma=0.149

[Tune-y] 3: mae.test.mean=4.4689978,rmse.test.rmse=7.1403968,rsq.test.mean=0.9688980; time: 3.4 min

[Tune-x] 4: kernel=radial; cost=0.463; gamma=0.653

[Tune-y] 4: mae.test.mean=4.5201345,rmse.test.rmse=7.6891054,rsq.test.mean=0.9639079; time: 2.7 min

[Tun


=== Hiperparámetros Óptimos Encontrados para SVM ===
$kernel
[1] "radial"

$cost
[1] 2.434821

$gamma
[1] 0.1675649


Modelo SVM entrenado exitosamente y listo para el ruedo.


In [8]:
# ---------------------------------------------------------------------
# 7. Evaluación del Modelo SVM en Test Data (Normalizado)
# ---------------------------------------------------------------------
cat("\n=== EVALUACIÓN EN CONJUNTO DE PRUEBA (TEST DATA) ===\n")

# Predicción usando el modelo SVM y el Task normalizado
svm_predictions <- predict(final_svm_model, test_task_norm)

# Extraemos reales y predichos
reales_svm <- svm_predictions$data$truth
predichos_svm <- svm_predictions$data$response

# Cálculo de métricas
mae_svm <- mean(abs(reales_svm - predichos_svm))
rmse_svm <- sqrt(mean((reales_svm - predichos_svm)^2))

# R-cuadrado
ss_total_svm <- sum((reales_svm - mean(reales_svm))^2)
ss_residual_svm <- sum((reales_svm - predichos_svm)^2)
rsq_svm <- 1 - (ss_residual_svm / ss_total_svm)

# TABLA DE POSICIONES FINAL DEL TORNEO
cat("--- TABLA DE POSICIONES DEL TORNEO (ACTUALIZADA) ---\n")
cat("1. Baseline (MLR)       -> MAE: 4.6900 min | RMSE: 7.8500 min | R²: 96.90%\n")
cat(sprintf("?. SVM (Radial)         -> MAE: %.4f min | RMSE: %.4f min | R²: %.2f%%\n", mae_svm, rmse_svm, rsq_svm * 100))
cat("3. k-NN (k=35)          -> MAE: 5.0035 min | RMSE: 8.1025 min | R²: 95.96%\n")
cat("4. CART (Árbol)         -> MAE: 5.5703 min | RMSE: 8.6955 min | R²: 95.34%\n")


=== EVALUACIÓN EN CONJUNTO DE PRUEBA (TEST DATA) ===
--- TABLA DE POSICIONES DEL TORNEO (ACTUALIZADA) ---
1. Baseline (MLR)       -> MAE: 4.6900 min | RMSE: 7.8500 min | R²: 96.90%
?. SVM (Radial)         -> MAE: 4.7312 min | RMSE: 7.8272 min | R²: 96.23%
3. k-NN (k=35)          -> MAE: 5.0035 min | RMSE: 8.1025 min | R²: 95.96%
4. CART (Árbol)         -> MAE: 5.5703 min | RMSE: 8.6955 min | R²: 95.34%


### Discusión de Resultados: SVM vs. Baseline

La evaluación de la Máquina de Soporte Vectorial (con kernel RBF) sobre el conjunto de prueba aislado arrojó un Error Absoluto Medio (MAE) de **4.7312 minutos** y un R² del **96.23%**. Si bien este modelo demostró ser el competidor no paramétrico más robusto del torneo (superando holgadamente a k-NN y CART), no logró quebrar la barrera de los **4.6900 minutos** establecida por el modelo paramétrico Baseline (MLR).

**Justificación Matemática y Fenomenológica:**
La incapacidad de un algoritmo de alta complejidad como SVM para superar a una regresión lineal múltiple en este escenario subraya la eficacia extrema de la Ingeniería de Características aplicada en la Fase 1. La variable sintética `ritmo_promedio_30k` destiló la varianza del sistema en una relación monótona y estrictamente lineal. En consecuencia, el mapeo a dimensiones superiores (característico del *Kernel Trick* radial del SVM) intentó modelar curvaturas y variaciones locales que, en este espacio de características linealizado, representaban mayoritariamente ruido estocástico (*stochastic noise*) en lugar de un patrón subyacente real. El hiperplano global del modelo paramétrico resultó ser la frontera de respuesta óptima.

---

## Paso 4: Meta-Aprendizaje y Modelos de Conjunto (Ensembles)

Tras evidenciar que los algoritmos de aprendizaje individuales (tanto paramétricos como no paramétricos) han alcanzado su límite representacional frente al Baseline, la teoría del aprendizaje estadístico sugiere una última línea de ataque: el **Meta-Aprendizaje**. 

En lugar de confiar en un único modelo maestro, los métodos de conjunto (*Ensembles*) combinan matemáticamente las predicciones de múltiples "aprendices base" para reducir la varianza, el sesgo o ambos. Nuestro cuarto y último contendiente será el **Bosque Aleatorio (Random Forest)**, un algoritmo que emplea la técnica de *Bagging* (Bootstrap Aggregating). Este ensamble entrena cientos de árboles de regresión (CART) de manera independiente sobre submuestras aleatorias del conjunto de datos, integrando además estocasticidad en la selección de características (*Feature Randomness*) en cada división nodal. La predicción final emerge del promedio aritmético de las predicciones de la multitud de árboles, mitigando severamente el problema de sobreajuste topológico que penalizó a nuestro modelo CART individual en el Paso 1.

In [ ]:
# =====================================================================
# Contendiente 4: Meta-Aprendizaje - Bosque Aleatorio (Random Forest)
# =====================================================================

# Instalamos y cargamos el motor subyacente si no lo tienes
# install.packages("randomForest")
library(randomForest)

cat("\n--- 1. Definición del Aprendiz Random Forest ---\n")
# Utilizamos regr.randomForest. Los ensambles basados en árboles NO requieren datos normalizados,
# por lo que volveremos a usar nuestro train_task original (sin estandarizar).
learner_rf <- makeLearner("regr.randomForest")

cat("\n--- 2. Espacio de Búsqueda de Hiperparámetros ---\n")
# ntree: Número de árboles (lo dejaremos fijo en 500 por velocidad, es un estándar robusto en la industria)
# mtry: Variables evaluadas en cada división nodal (muy importante calibrar)
# nodesize: Tamaño mínimo de los nodos terminales (controla la profundidad implícita)
rf_param_space <- makeParamSet(
  makeIntegerParam("mtry", lower = 1, upper = 3), # Tenemos pocas variables, probaremos de 1 a 3
  makeIntegerParam("nodesize", lower = 5, upper = 20)
)

tune_control_rf <- makeTuneControlGrid()

cat("\n--- 3. Sintonización de Hiperparámetros (Grid Search RF) ---\n")
cat("Entrenando cientos de árboles... espera un momento.\n")

tuned_rf <- tuneParams(
  learner = learner_rf,
  task = train_task, # Usamos la tarea original, sin estandarizar
  resampling = cv_strategy, 
  par.set = rf_param_space,
  control = tune_control_rf,
  measures = list(mae, rmse, rsq)
)

cat("\n=== Hiperparámetros Óptimos Encontrados para Random Forest ===\n")
print(tuned_rf$x)

# --- 4. Entrenamiento del Ensamble Definitivo ---
# Modificamos el aprendiz para incluir los parámetros óptimos y fijamos ntree en 500
final_learner_rf <- setHyperPars(learner_rf, par.vals = c(tuned_rf$x, ntree = 500))
final_rf_model <- train(final_learner_rf, train_task)

cat("\nBosque Aleatorio ensamblado exitosamente.\n")